In [74]:
import os
from openai import AzureOpenAI
from tqdm import tqdm
import time
import re
from sklearn.metrics import *
import krippendorff

In [75]:
!nvidia-smi

# Define Endpoint and Query Function

In [76]:
api_key = os.getenv("AZURE_OPENAI_API_KEY_MODEL")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_MODEL")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION_MODEL")

In [77]:
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=api_key,
    api_version=api_version,
)

In [78]:
response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

In [79]:
print(response.choices[0].message.content)

In [80]:
# import os
# from openai import AzureOpenAI

# endpoint = ""
# model_name = "gpt-5.2-chat"
# deployment_name = "gpt-5.2-chat"

# subscription_key = ""
# api_version = "2024-12-01-preview"

# client = AzureOpenAI(
#     api_version=api_version,
#     azure_endpoint=endpoint,
#     api_key=subscription_key,
# )

# response = client.chat.completions.create(
#     messages=[
#         {
#             "role": "system",
#             "content": "You are a helpful assistant.",
#         },
#         {
#             "role": "user",
#             "content": "I am going to Paris, what should I see?",
#         }
#     ],
#     max_completion_tokens =16384,
#     model=deployment_name
# )

# print(response.choices[0].message.content)

In [81]:
def query_llm(sys_msg, query_msg, dt=1e-2):
    time.sleep(dt)
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": query_msg}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(e)
        return -1

# Load LLM Response Dataset

In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [85]:
test_path = "sps_dataset_test_split.csv"

In [86]:
all_dfs = [pd.read_csv(test_path)]

# Query for LLM

In [87]:
all_criteria = ['Completeness', 'Clarity', 'Interpretability', 'Conciseness', 'Accuracy', 'Relevance']

criteria_prompts = {}
criteria_prompts["Completeness"] = """
Completeness is a criteria that judges whether the response is complete and comprehensive. 
That means a good response should have included all of the relevant information that answers the question.
Conversely, a poor response would miss out important details relevant to the question.
This is the exact scoring rubrics:
-2: "Important information is missing, causing major misunderstandings.",
-1: "Several details are missing, making the response only partially usable.",
0: "Mostly complete but lacking a few supporting details.",
1: "Complete with all necessary information and minimal omissions.",
2: "Fully comprehensive with all required details and no omissions."
"""
criteria_prompts["Clarity"] = """
Clarity is a criteria that judges whether the response is clear and easy to understand. 
That means a good response should be easily understandable and well phrased.
Conversely, a bad response would be poorly phrased with lack of focus and hence not easily understandable.
This is the exact scoring rubrics:
-2: "Very unclear and confusing, making it hard to understand.",
-1: "Partially unclear with awkward wording or ambiguous sentences.",
0: "Somewhat clear but with minor ambiguity or weak phrasing.",
1: "Clear, easy to follow, and well-phrased.",
2: "Extremely clear, well-articulated, and highly readable."
"""
criteria_prompts["Interpretability"] = """
Interpretability is a criteria that judges whether the response is interpretable and have a clear logic.
That means a good response should be self-consistent while having a logic that is easy to understand.
Conversely, a bad response would be very convoluted and difficult to understand, even if the logic is correct.
This is the exact scoring rubrics:
-2: "Difficult to understand with tangled reasoning or unclear logic.",
-1: "Partially understandable but with unclear logic or weak organization.",
0: "Generally understandable but occasionally confusing or inconsistent.",
1: "Easy to understand with clear logic and strong organization.",
2: "Extremely easy to understand, logically strong, and excellently organized."
"""
criteria_prompts["Conciseness"] = """
Conciseness is a criteria that judges whether the response is concise in writing.
That means a good response should be not contain any redundant phrasing or language.
Conversely, a bad resonse would be contain irrelevant words that inflates the length of the response. 
A bad response can usually be shortened by re-writing the response to have more concise English.
This is the exact scoring rubrics:
-2: "Very wordy, redundant, or filled with unnecessary details.",
-1: "Somewhat verbose with noticeable redundancy.",
0: "Some unnecessary wording but overall acceptable length.",
1: "Concise with minimal redundancy and clear expression.",
2: "Highly concise, focused, and free of all unnecessary words."
"""
criteria_prompts["Accuracy"] = """
Accuracy is a criteria that judges whether the response is accurate in content and does not contain false or misleading information.
That means a good response should be fully accurate with all of the information provided found in the given context.
A good response should not contain information not given in the context, even if they are common knowledge.
A good response should also avoid misleading statements, or statements that imply things that are not given in the context.
Conversely, a bad response would include incorrect or misleading statements, or statements that are based on pre-existing knowledge but not supported by the context.
This is the exact scoring rubrics:
-2: "Contains factually incorrect or fabricated information.",
-1: "Contains several factual inaccuracies or unclear claims.",
0: "Mostly accurate but with minor errors or ambiguous statements.",
1: "Accurate and reliable with no significant factual issues.",
2: "Fully precise, factually correct, and verifiable throughout."
"""
criteria_prompts["Relevance"] = """
Relevance is a criteria that judges whether the response is relevant to the question being asked.
That means a good response should be avoid unnecessary details that the question didn't ask about, even if they might be tangentially relevant.
A good response should not contain statements that are irrelevant to the question being asked, even if they are true and provided in the context.
Conversely, a bad response would include irrelevant statements, even if they are true based on the context.
This is the exact scoring rubrics:
-2: "Content is mostly irrelevant or off-topic.",
-1: "Content is partially irrelevant or only loosely connected to the topic.",
0: "Content is somewhat relevant but contains unnecessary or unfocused parts.",
1: "Content is relevant and contributes meaningfully to the topic.",
2: "Content is highly relevant, targeted, and fully aligned with the topic."
"""

In [88]:
def get_llm_input(text_chunk, question, answer):
    """
    Returns the system and query text to score the response
    """
    
    # Filter out non-ASCII characters
    text_chunk = re.sub(r'[^\x00-\x7f]',r'', text_chunk)
    question = re.sub(r'[^\x00-\x7f]',r'', question)
    # answer = re.sub(r'[^\x00-\x7f]',r'', answer)

    system_prompt = \
    f"""You are tasked with scoring a response to a question based on a context. You will score it based on 6 criterias.
    Your will be given a context, a question, and an answer. All three of them will be in text format.

    These are the 6 criterias: 
    {criteria_prompts["Completeness"]}
    {criteria_prompts["Clarity"]}
    {criteria_prompts["Interpretability"]}
    {criteria_prompts["Conciseness"]}
    {criteria_prompts["Accuracy"]}
    {criteria_prompts["Relevance"]}
    
    Finally, output only the score.

    For example, below is a possible set of input:
    This is the context given: Company A was created in 1992.
    This is the question asked: Which year was company A created?
    This is the given answer: Company A was created in year 1992 by Mr X, driven by the mission to achieve Y.
    
    This is a possible output:
    -1 2 0 1 1 -2
    """

    query_prompt = \
    f"""This is the context given: {text_chunk}
    This is the question asked: {question}
    This is the given answer: {answer}
    Score the answer based on the 6 given criterias. Output only the scores, separated by whitespace. Output nothing else. THe output should be 6 numbers.
    """
    return system_prompt, query_prompt

In [89]:
sys_msg, q_msg = get_llm_input(all_dfs[0]['text_chunk'][1], all_dfs[0]['question'][1], all_dfs[0]['generated_answer'][1])

In [90]:
sys_msg

In [91]:
q_msg

In [92]:
response = query_llm(sys_msg=sys_msg, query_msg=q_msg)
response

# Generating and Exporting to npy

In [93]:
all_scores = []

In [94]:
all_dfs[0].head()

In [96]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from tqdm import tqdm
import time

# reuse your query_llm as-is (remove the test sleep dt)
# def query_llm(sys_msg, query_msg, dt=1e-2): ...

def parse_response_text(text):
    """Parse LLM response into six ints, fallback to 2 on error."""
    if isinstance(text, int) and text == -1:
        return [2]*6
    parts = str(text).split()
    out = [2]*6
    for j in range(min(6, len(parts))):
        try:
            out[j] = int(parts[j])
        except Exception:
            out[j] = 2
    return out

async def run_queries_async(df, max_workers=10, retries=2):
    """
    df: dataframe like your df
    max_workers: number of threads to run in parallel (tuneable)
    retries: per-item retry attempts for transient errors
    """
    loop = asyncio.get_running_loop()
    executor = ThreadPoolExecutor(max_workers=max_workers)
    results = [None] * df.shape[0]
    tasks = []

    async def worker(i):
        sys_msg, q_msg = get_llm_input(df['text_chunk'][i],
                                       df['question'][i],
                                       df['generated_answer'][i])
        attempt = 0
        while True:
            attempt += 1
            try:
                # run the blocking query in a thread
                func = partial(query_llm, sys_msg, q_msg)
                text = await loop.run_in_executor(executor, func)
                results[i] = parse_response_text(text)
                return  # success
            except Exception as e:
                # query_llm already prints e and returns -1 on exception,
                # but we guard here in case other exceptions arise.
                if attempt > retries:
                    results[i] = [2]*6
                    return
                # small backoff (blocking sleep happens in thread; here we await)
                await asyncio.sleep(0.5 * attempt)

    # create tasks
    for i in range(df.shape[0]):
        tasks.append(asyncio.create_task(worker(i)))

    # update progress as tasks finish
    with tqdm(total=len(tasks)) as pbar:
        for finished in asyncio.as_completed(tasks):
            await finished
            pbar.update(1)

    executor.shutdown(wait=True)
    return results

In [97]:
df = all_dfs[0]
# Tune max_workers — start with 5-20 depending on API limits/network.
scores = await run_queries_async(df, max_workers=20, retries=2)
all_scores.append(scores)

In [98]:
all_scores = np.array(all_scores)
all_scores